# Creating 100 simplified versions of the pie plot (Pop/Latin only)

Simplified variant of `100_versions_pie_plot_and_posts.ipynb`, built 2026-07-16 to test a supervisor hypothesis: the near-50% diagonal accuracy several models show in the main E1 study (`docs/THESIS.md` Section 6.1) might partly reflect the 9-slice chart being visually complex to parse, rather than genuine indifference to correctness.

**Design decision (confirmed with the user 2026-07-16):** Pop stays at 23.5% and Latin stays at 11.0% — identical to the original chart, so the post text, claim-correctness framing, and ground truth all still apply unchanged. The other 7 genres are merged into a single unlabeled "Other" wedge (65.5%) with no percentage shown — only Pop's and Latin's values are ever visible on the chart. Everything else (color palette pool, per-image random palette/shuffle, explode, start angle, pctdistance, line width, font sizes, title) is kept exactly as random and as varied as the original 100-version generator, so this is a like-for-like simplification of the *same* generation process, not a differently-styled chart.

**Note on which notebook this is actually simplifying:** an earlier attempt at this task incorrectly assumed `utils/chart_creation.ipynb` (a single fixed chart) was the real source, based on `ground_truth_all.csv` showing identical colors ("green"/"blue") for all 100 images. That ground truth field turned out to only ever have been checked against image `001` specifically (confirmed: `benchmarking/utils/quanti_benchmarking_1_details.py`'s color questions are only ever run against image `001`, never the other 99) — and a direct visual comparison of images `001` and `002` confirmed colors genuinely vary a lot per image. So *this* notebook (`100_versions_pie_plot_and_posts.ipynb`) — with its per-image random palette selection — is the real basis for the simplified version, not `chart_creation.ipynb`.

In [ ]:
import matplotlib.pyplot as plt
import random
import os

# Fixed values — unchanged from the original chart, so the post text and ground truth
# ("Pop was more popular than Latin") stay valid without any changes.
fixed_pop = 23.5
fixed_latin = 11.0
fixed_other = round(100 - fixed_pop - fixed_latin, 1)  # 65.5 — every other genre, merged into one wedge

labels = ["Pop", "Latin", "Other"]

# Same color palettes as the original 9-slice generator — Pop and Latin each still draw a random
# color from a randomly-chosen, shuffled palette per image, preserving the same visual variety
# across the 100 versions. "Other" always uses a fixed neutral gray (already one of the original
# palette's own colors, used for its own "Other" genre) so the merged wedge never looks interesting
# or draws attention away from Pop/Latin.
color_palettes = [
    ["#1DB954", "#191414", "#e74c3c", "#f39c12", "#3498db", "#9b59b6", "#1abc9c", "#bdc3c7", "#e0e0e0"],
    ["#FF6B6B", "#4ECDC4", "#45B7D1", "#96CEB4", "#FFEAA7", "#DDA0DD", "#98D8C8", "#F7DC6F", "#BB8FCE"],
    ["#2C3E50", "#E74C3C", "#ECF0F1", "#3498DB", "#2ECC71", "#F39C12", "#9B59B6", "#1ABC9C", "#E67E22"],
    ["#FF9A9E", "#FAD0C4", "#A18CD1", "#FBC2EB", "#84FAB0", "#8FD3F4", "#D4FC79", "#96E6A1", "#FFDDE1"],
    ["#667EEA", "#764BA2", "#F093FB", "#F5576C", "#4FACFE", "#00F2FE", "#43E97B", "#38F9D7", "#FA709A"],
]
OTHER_COLOR = "#e0e0e0"

# Dark colors that need white percentage text — same list as the original
dark_colors = {
    "#191414", "#000000", "#2c3e50", "#000", "#667eea", "#764ba2",
    "#191414", "#3498db", "#9b59b6", "#4facfe", "#f5576c"
}

os.makedirs("100_pie_charts_simple", exist_ok=True)

for i in range(100):
    random.seed(i)

    sizes = [fixed_pop, fixed_latin, fixed_other]

    if i == 0:
        # Image 001 is the one example used for Phase 1 quantitative benchmarking's color
        # questions (benchmarking/utils/quanti_benchmarking_1_details.py only ever tests image
        # 001 - confirmed 2026-07-16). Forcing its colors here, rather than trusting the random
        # draw below, guarantees it keeps Pop=green/Latin=blue (matching the real chart's image
        # 001 and ground_truth_all.csv) - confirmed 2026-07-17 the random draw is NOT reliable for
        # this (Latin came out light pink on one actual run), since this notebook draws fewer
        # random numbers per image than the original 9-slice generator (no genre-size
        # randomization), so the two notebooks' random-call sequences do not stay in sync.
        colors = ["#1DB954", "#3498db", OTHER_COLOR]  # Pop=green, Latin=blue, same hex as the original
    else:
        # Random palette + shuffle, same as the original — take the first two shuffled colors for
        # Pop/Latin, skipping OTHER_COLOR itself if the shuffle happens to hand it to one of them
        # (one of the original palettes includes #e0e0e0 as one of its 9 colors — without this filter,
        # Latin or Pop could randomly end up the same gray as the merged "Other" wedge, defeating the
        # point of a clearer chart; caught 2026-07-16 via a dry run of this exact logic before plotting).
        palette = random.choice(color_palettes)[:]
        random.shuffle(palette)
        candidate_colors = [c for c in palette if c.lower() != OTHER_COLOR.lower()]
        colors = [candidate_colors[0], candidate_colors[1], OTHER_COLOR]

    startangle = random.randint(0, 360)

    # Explode only ever applies to Pop or Latin — exploding the merged "Other" wedge wouldn't mean anything
    explode = [0.0, 0.0, 0.0]
    explode[random.randint(0, 1)] = random.choice([0.0, 0.05, 0.1])
    explode = tuple(explode)

    pctdistance = random.uniform(0.70, 0.88)
    figsize = random.choice([(6, 6), (7, 7), (6.5, 6.5)])

    fig, ax = plt.subplots(figsize=figsize)

    def autopct_fn(pct, _pop=fixed_pop, _latin=fixed_latin):
        # Only Pop and Latin ever print a percentage — the merged "Other" wedge stays unlabeled.
        # This is the entire point of the simplified chart: only 2 numbers to read, not 9.
        if abs(pct - _pop) < 0.05 or abs(pct - _latin) < 0.05:
            return f"{pct:.1f}%"
        return ""

    wedges, texts, autotexts = ax.pie(
        sizes,
        labels=labels,
        colors=colors,
        explode=explode,
        autopct=autopct_fn,
        startangle=startangle,
        pctdistance=pctdistance,
        wedgeprops=dict(edgecolor="white", linewidth=random.uniform(1.0, 2.5))
    )

    fontsize = random.randint(8, 11)
    for k, text in enumerate(autotexts):
        text.set_fontsize(fontsize)
        text.set_fontweight("bold")
        if colors[k].lower() in dark_colors:
            text.set_color("white")
        else:
            text.set_color("black")

    ax.set_title("Music Genre Preferences of Spotify Users",
                 fontsize=random.randint(11, 14),
                 fontweight="bold", pad=20)

    plt.tight_layout()
    plt.savefig(f"100_pie_charts_simple/spotify_genre_pie_chart_simple_{i+1:03d}.png", dpi=150, bbox_inches="tight")
    plt.close()
    print(f"Saved simplified chart {i+1}")

print("Done!")


## Next steps (not done in this notebook)

This notebook only produces the 100 standalone simplified chart PNGs, in `100_pie_charts_simple/`. Wiring them into full Facebook-post images for an actual E1 pilot run would still need the same post-assembly step used for the original charts (HTML/CSS templating via `utils/updated_post_generator_all_visible_emojis.ipynb`, then headless-Chrome rasterization via `utils/html_to_png.ipynb`), substituting each `spotify_genre_pie_chart_simple_{i:03d}.png` for the corresponding original chart image — not attempted here, since `docs/THESIS.md` Chapter 8's simpler-chart suggestion only calls for a pilot on one model, not a full 100-image regeneration.